**Human Movement Intelligence | Presented by: Reza Saadatyar (2026)**

**12 | Development Model Selection and Protocol Specification**

![HARTH Development model selection and protocol workflow](../../images/12_consolidated_workflow.svg)

**Research objective:** Integrate the participant-disjoint seven-model benchmark from [Notebook 10](10_harth_advanced_temporal_architectures_and_matched_benchmark.ipynb) with the multi-seed and class-imbalance analysis from [Notebook 11](11_harth_multiseed_stability_and_imbalance_robustness.ipynb). Describe the source-grounded selection of a primary 1D-ResNet and a fixed Random Forest reference, and document the existing training and evaluation protocol without Test-driven selection.

**Development evidence:** Both preceding studies use the pinned HARTH v2.0 source, six acceleration channels, eight activity labels and **14,201 label-pure windows from 29 Development participants** under a label-independent cap of 500 windows per person. Notebook 10 compares **seven models over three participant-grouped folds**. Notebook 11 compares **four model/loss conditions across three seeds and the same three folds (36 model–fold results)**.

**Recorded decision:** 1D-ResNet with inverse-square-root weighted Cross-Entropy is the primary configuration, selected on Development evidence (mean Macro F1 **0.8336** for seed 42 in the seven-model benchmark and **0.8289** across three seeds in the robustness study). Random Forest remains the fixed classical reference; its pooled Shuffling F1 (**0.5032**) exceeds the seed-42 ResNet value (**0.3969**). Seed 42 is retained as the recorded reproducibility convention, not selected by inspecting Test performance.

**Recorded procedure:** The 1D-ResNet uses Adam, learning rate 0.001, batch size 128, 400 updates, Development-fitted channel statistics and the final-step checkpoint. Random Forest uses 30 deterministic signal features, 120 trees, `min_samples_leaf=3` and balanced-subsample class weighting. The fixed eight-class Macro F1 and class/participant reporting rules are documented below.

**Evidence and boundaries:** This notebook reads saved `10_` and `11_` CSV files and directly records the established source, model and evaluation settings. It does not load sensor recordings, train models or evaluate Test participants. S023/S032 were inspected descriptively earlier in the study, so their subsequent same-study assessment is qualified rather than blind external validation. The recorded settings retain the established Development decision; they do not introduce Test-informed selection.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")

if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("wearable_activity_recognition")

# ----------------- 2. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 3. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------ 4. Study Utilities and Stage-Owned Output Directory ------------
from core.output_download import download_outputs_from_colab

output_dir = root / "wearable_activity_recognition" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
SOURCE_CLASSES = tuple(range(1, 9))
FIXED_PRIMARY = "1D-ResNet"
FIXED_REFERENCE = "Random Forest"

print("Development selection source:", output_dir)
print("Scientific settings: recorded in Notebook 12 and its output tables")


**02. Read Matched Development Evidence**

Read the current CSV evidence from Notebook 10 (seven models, eight-class class metrics, paired folds and cohort support) and Notebook 11 (four architecture/loss conditions, 36 held-out model–fold records, class-level scores and the same Development cohort). No source recordings, saved model checkpoints or reserved Test participants are accessed.

The input columns and unique model/fold/seed identities are checked before comparison. The two Development cohort summaries are compared at the participant and window-count level. Agreement supports consistent recorded data selection but does not independently reproduce it from raw recordings.

In [3]:
# =================== 02. Read Matched Development Evidence ===================

# --------------- 1. Load the Existing Notebook 10 and 11 CSVs ----------------
source_files = {
    "benchmark": "10_matched_cv_model_summary.csv",
    "benchmark_classes": "10_matched_cv_class_metrics.csv",
    "benchmark_folds": "10_matched_cv_fold_metrics.csv",
    "benchmark_cohort": "10_development_cohort_summary.csv",
    "benchmark_support": "10_development_activity_support.csv",
    "benchmark_fold_design": "10_matched_fold_design.csv",
    "robustness": "11_robustness_model_summary.csv",
    "robustness_folds": "11_robustness_fold_metrics.csv",
    "robustness_classes": "11_robustness_class_metrics.csv",
    "robustness_cohort": "11_development_cohort_summary.csv",
    "paired_loss": "11_resnet_paired_loss_scores.csv",
}
missing = [
    filename for filename in source_files.values()
    if not (output_dir / filename).is_file()
]
if missing:
    raise FileNotFoundError("Development evidence missing: " + ", ".join(missing))
tables = {
    key: pd.read_csv(output_dir / filename)
    for key, filename in source_files.items()
}

# -------------------- 2. Check Essential Table Interfaces --------------------
required_columns = {
    "benchmark": {"model", "macro_f1_mean", "macro_f1_sd"},
    "benchmark_classes": {"model", "class_code", "support", "precision", "recall", "f1"},
    "benchmark_folds": {"model", "fold", "macro_f1", "validation_windows"},
    "benchmark_cohort": {"subject_id", "sampled_windows"},
    "benchmark_support": {"activity", "windows"},
    "benchmark_fold_design": {"fold", "validation_windows", "validation_subject_ids"},
    "robustness": {"architecture", "loss_method", "mean_across_seed_fold_macro_f1", "across_seed_sd"},
    "robustness_folds": {"architecture", "loss_method", "seed", "fold", "macro_f1", "validation_windows"},
    "robustness_classes": {"architecture", "loss_method", "seed", "class_code", "support", "precision", "recall", "f1"},
    "robustness_cohort": {"subject_id", "sampled_windows"},
    "paired_loss": {"seed", "fold", "focal_minus_weighted", "class_balanced_minus_weighted"},
}
for key, required in required_columns.items():
    absent = required.difference(tables[key].columns)
    if absent:
        raise ValueError(f"{source_files[key]} lacks: {sorted(absent)}")

# ------------ 3. Reconcile Cohort Coverage Across the Two Studies ------------
cohort_10 = tables["benchmark_cohort"].sort_values("subject_id").reset_index(drop=True)
cohort_11 = tables["robustness_cohort"].sort_values("subject_id").reset_index(drop=True)
if not cohort_10.equals(cohort_11):
    raise ValueError("Notebook 10 and 11 Development cohort summaries differ.")
development_windows = int(cohort_10["sampled_windows"].sum())
development_people = int(cohort_10["subject_id"].nunique())
if {"S023", "S032"} & set(cohort_10["subject_id"]):
    raise ValueError("Reserved participants appear in the Development cohort.")
if int(tables["benchmark_support"]["windows"].sum()) != development_windows:
    raise ValueError("Class support does not reconcile with Development windows.")

print("Development participants:", development_people)
print("Development source windows:", development_windows)
print("Model summaries:", len(tables["benchmark"]))
print("Robustness fold records:", len(tables["robustness_folds"]))
print("Identical participant/source counts in the two cohort tables.")


Development participants: 29
Development source windows: 14201
Model summaries: 7
Robustness fold records: 36
Identical participant/source counts in the two cohort tables.


**03. Compare Seven Models and Minority-Class Performance**

Notebook 10 evaluates all seven models on identical participant-grouped Development folds. The fixed-eight-class Macro F1 mean identifies overall performance, while pooled out-of-fold Shuffling F1 provides a class-specific comparison. The two metrics have different aggregation semantics and can favour different models.

The reported three-fold Macro F1 mean is **0.8336** for 1D-ResNet, **0.8138** for TCN and **0.8094** for Random Forest. Random Forest instead has the highest Shuffling F1 (**0.5032**) versus **0.3969** for 1D-ResNet. A joint plot shows this trade-off without reproducing Notebook 10's complete set of ranking, confusion and class-level figures.

In [ ]:
# ========== 03. Compare Seven Models and Minority-Class Performance ==========

# ---------------- 1. Assemble the Seven-Model Decision Table -----------------
benchmark = tables["benchmark"].copy()
classes = tables["benchmark_classes"].copy()
folds_10 = tables["benchmark_folds"].copy()
expected_models = {
    "1D-ResNet", "TCN", "Random Forest", "Standard CNN",
    "Weighted CNN", "Multi-Scale CNN", "Time-Series Transformer",
}
if (benchmark["model"].duplicated().any() or
        set(benchmark["model"]) != expected_models):
    raise ValueError("Seven-model comparison has missing or duplicate families.")
if classes.duplicated(["model", "class_code"]).any():
    raise ValueError("Duplicate model/activity class metrics.")
for model, group in classes.groupby("model"):
    if set(group["class_code"]) != set(SOURCE_CLASSES):
        raise ValueError(f"Missing class-level results for {model}.")
if folds_10.duplicated(["model", "fold"]).any():
    raise ValueError("Duplicated matched model–fold records.")
if not all(set(g["fold"]) == {1, 2, 3}
           for _, g in folds_10.groupby("model")):
    raise ValueError("Each benchmark model must have three matched folds.")

shuffling = classes.loc[
    classes["class_code"].eq(3),
    ["model", "support", "precision", "recall", "f1"]
].rename(columns={
    "support": "shuffling_support",
    "precision": "shuffling_precision",
    "recall": "shuffling_recall",
    "f1": "shuffling_pooled_f1",
})
selection_comparison = benchmark.merge(
    shuffling, on="model", validate="one_to_one"
).sort_values("macro_f1_mean", ascending=False)
display(selection_comparison[[
    "model", "macro_f1_mean", "macro_f1_sd",
    "shuffling_support", "shuffling_pooled_f1"
]].round(4))
selection_comparison.to_csv(
    output_dir / "12_development_model_selection_evidence.csv",
    index=False,
)

# --------- 2. Plot Two Complementary Development Selection Criteria ----------
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    selection_comparison["macro_f1_mean"],
    selection_comparison["shuffling_pooled_f1"],
    s=70, zorder=3,
)
# ---------------- 3. Position Model Labels Without Overlap -----------------
# Place neighbouring TCN and Transformer labels on opposite sides.
label_offsets = {
    "1D-ResNet": (-10, 13),
    "TCN": (11, -13),
    "Random Forest": (7, 10),
    "Time-Series Transformer": (-10, 14),
    "Multi-Scale CNN": (7, 8),
    "Weighted CNN": (8, 8),
    "Standard CNN": (8, 8),
}
for _, row in selection_comparison.iterrows():
    model_name = row["model"]
    dx, dy = label_offsets.get(model_name, (8, 8))
    ax.annotate(
        model_name,
        (row["macro_f1_mean"], row["shuffling_pooled_f1"]),
        xytext=(dx, dy), textcoords="offset points", fontsize=9,
        ha="right" if dx < 0 else "left",
        va="top" if dy < 0 else "bottom",
        arrowprops=(
            {"arrowstyle": "-", "color": "0.5", "lw": 0.7}
            if model_name in {"TCN", "Time-Series Transformer"}
            else None
        ),
    )
x_values = selection_comparison["macro_f1_mean"]
y_values = selection_comparison["shuffling_pooled_f1"]
ax.set_xlim(max(0, x_values.min() - 0.036), min(1, x_values.max() + 0.066))
ax.set_ylim(max(-0.05, y_values.min() - 0.05), min(1, y_values.max() + 0.067))
ax.set(
    xlabel="Mean three-fold Development Macro F1",
    ylabel="Pooled Development Shuffling F1",
    title="Model selection evidence | Overall versus Shuffling performance",
)
ax.grid(alpha=0.2)
fig.tight_layout()
save_figure(fig, output_dir / "12_overall_vs_shuffling_selection.png")
plt.show()


**04. Assess Multi-Seed and Loss Robustness**

Notebook 11 adds three initialisation seeds to Weighted CE 1D-ResNet and TCN, then compares Focal and Class-Balanced Cross-Entropy on the same ResNet architecture and participant folds. Across nine matched model–fold results per condition, the mean Macro F1 values are **0.8289** for Weighted CE ResNet, **0.8106** for Weighted CE TCN, **0.8057** for Focal ResNet and **0.7915** for Class-Balanced ResNet.

The three-fold seed-42 ResNet and TCN scores reproduce the corresponding Notebook 10 results. This exact cross-stage comparison is checked below. A loss difference is paired within the same seed and fold; a three-seed SD is descriptive initialisation variation, not a population confidence interval. The original full loss-ablation figure remains in Notebook 11.

In [5]:
# ================= 04. Assess Multi-Seed and Loss Robustness =================

# ---------- 1. Recalculate Condition Means From the Saved 36 Folds -----------
robust = tables["robustness"].copy()
folds_11 = tables["robustness_folds"].copy()
condition_set = {
    ("1D-ResNet", "weighted_ce"),
    ("TCN", "weighted_ce"),
    ("1D-ResNet", "focal"),
    ("1D-ResNet", "class_balanced"),
}
observed_conditions = set(zip(
    robust["architecture"], robust["loss_method"]
))
if observed_conditions != condition_set or robust.duplicated(
    ["architecture", "loss_method"]
).any():
    raise ValueError("Incomplete or duplicated robustness conditions.")
if folds_11.duplicated(
    ["architecture", "loss_method", "seed", "fold"]
).any():
    raise ValueError("Duplicated robustness model–seed–fold results.")
seed_fold_grid = {
    (seed, fold) for seed in (42, 123, 2026) for fold in (1, 2, 3)
}
for key, group in folds_11.groupby(["architecture", "loss_method"]):
    if set(zip(group["seed"], group["fold"])) != seed_fold_grid:
        raise ValueError(f"Incomplete seed–fold grid for {key}.")
    saved_mean = robust.loc[
        robust["architecture"].eq(key[0]) &
        robust["loss_method"].eq(key[1]),
        "mean_across_seed_fold_macro_f1"
    ].iloc[0]
    if not np.isclose(group["macro_f1"].mean(), saved_mean, atol=1e-10):
        raise ValueError(f"Robustness summary differs from its folds: {key}.")

# -------------- 2. Check Seed-42 Baselines Against Notebook 10 ---------------
baseline_11 = folds_11.loc[
    folds_11["loss_method"].eq("weighted_ce") &
    folds_11["seed"].eq(42) &
    folds_11["architecture"].isin(["1D-ResNet", "TCN"]),
    ["architecture", "fold", "macro_f1"]
].rename(columns={"architecture": "model", "macro_f1": "robustness_f1"})
baseline_10 = folds_10.loc[
    folds_10["model"].isin(["1D-ResNet", "TCN"]),
    ["model", "fold", "macro_f1"]
].rename(columns={"macro_f1": "benchmark_f1"})
paired_baseline = baseline_10.merge(
    baseline_11, on=["model", "fold"], validate="one_to_one"
)
if len(paired_baseline) != 6 or not np.allclose(
    paired_baseline["benchmark_f1"],
    paired_baseline["robustness_f1"], atol=1e-10, rtol=0,
):
    raise ValueError("Seed-42 cross-stage baseline fold results differ.")

robustness_comparison = robust.sort_values(
    "mean_across_seed_fold_macro_f1", ascending=False
)
display(robustness_comparison.round(4))
robustness_comparison.to_csv(
    output_dir / "12_development_robustness_selection_evidence.csv",
    index=False,
)

# ---------------- 3. Describe Paired ResNet Loss Differences -----------------
paired_loss = tables["paired_loss"].copy()
if paired_loss.duplicated(["seed", "fold"]).any() or set(zip(
    paired_loss["seed"], paired_loss["fold"]
)) != seed_fold_grid:
    raise ValueError("Paired ResNet loss comparisons lack a seed/fold.")
for column in ("focal_minus_weighted", "class_balanced_minus_weighted"):
    print(
        column, "| mean paired difference:",
        round(float(paired_loss[column].mean()), 4),
        "| positive pairs:", int((paired_loss[column] > 0).sum()),
        "of", len(paired_loss),
    )
paired_loss.to_csv(
    output_dir / "12_resnet_paired_loss_evidence.csv", index=False
)


,architecture,loss_method,mean_across_seed_fold_macro_f1,across_seed_sd,total_training_s
2,1D-ResNet,weighted_ce,0.8289,0.0053,31.7206
3,TCN,weighted_ce,0.8106,0.0078,35.6860
1,1D-ResNet,focal,0.8057,0.0051,32.4593
0,1D-ResNet,class_balanced,0.7915,0.0190,30.7574


focal_minus_weighted | mean paired difference: -0.0231 | positive pairs: 2 of 9
class_balanced_minus_weighted | mean paired difference: -0.0374 | positive pairs: 1 of 9


**05. Interpret Minority-Activity Loss Trade-offs**

The 500-cap cohort contains **219 Shuffling**, **192 Stairs ascending** and **182 Stairs descending** windows. Random Forest has the best pooled Shuffling F1 in Notebook 10 (**0.5032**), providing a meaningful classical comparison alongside the higher overall Macro F1 of 1D-ResNet.

Within the three-seed ResNet robustness study, Class-Balanced CE increases mean Shuffling Recall to **0.7686**, compared with **0.5464** for Weighted CE. However, its mean Shuffling Precision is **0.2309** versus **0.3041**, and its mean Shuffling F1 is lower (**0.3515** versus **0.3874**). The averaging unit for these robustness scores is the three seed-specific pooled OOF class metrics, not the original source windows treated as independent replications.

In [6]:
# ============== 05. Interpret Minority-Activity Loss Trade-offs ==============

# -------- 1. Summarise Robustness Minority-Class Precision and Recall --------
robust_classes = tables["robustness_classes"].copy()
if robust_classes.duplicated(
    ["architecture", "loss_method", "seed", "class_code"]
).any():
    raise ValueError("Duplicate robustness class rows.")
if not all(set(group["class_code"]) == set(SOURCE_CLASSES)
           for _, group in robust_classes.groupby(
               ["architecture", "loss_method", "seed"]
           )):
    raise ValueError("Eight-class robustness evidence is incomplete.")

minority = robust_classes.loc[
    robust_classes["class_code"].isin([3, 4, 5])
]
minority_summary = minority.groupby(
    ["architecture", "loss_method", "class_code"], as_index=False
).agg(
    class_support=("support", "first"),
    mean_precision_across_seeds=("precision", "mean"),
    mean_recall_across_seeds=("recall", "mean"),
    mean_f1_across_seeds=("f1", "mean"),
)
minority_summary["evidence"] = "Three-seed mean of pooled OOF metrics"
rf_shuffling = classes.loc[
    classes["model"].eq(FIXED_REFERENCE) & classes["class_code"].eq(3)
]
print("Random Forest pooled Shuffling F1 (Notebook 10):",
      round(float(rf_shuffling["f1"].iloc[0]), 4))
display(minority_summary.loc[
    minority_summary["class_code"].eq(3)
].round(4))
minority_summary.to_csv(
    output_dir / "12_minority_loss_tradeoffs.csv", index=False,
)


Random Forest pooled Shuffling F1 (Notebook 10): 0.5032


,architecture,loss_method,class_code,class_support,mean_precision_across_seeds,mean_recall_across_seeds,mean_f1_across_seeds,evidence
0,1D-ResNet,class_balanced,3,219,0.2309,0.7686,0.3515,Three-seed mean of pooled OOF metrics
3,1D-ResNet,focal,3,219,0.2721,0.6332,0.3794,Three-seed mean of pooled OOF metrics
6,1D-ResNet,weighted_ce,3,219,0.3041,0.5464,0.3874,Three-seed mean of pooled OOF metrics
9,TCN,weighted_ce,3,219,0.2654,0.4277,0.3261,Three-seed mean of pooled OOF metrics


**06. Document the Primary and Reference Selection**

The recorded Development-based selection identifies **1D-ResNet with inverse-square-root Weighted Cross-Entropy** as the primary model. It has the highest mean Macro F1 in the seven-model seed-42 benchmark (**0.8336**) and the highest average across the three-seed robustness conditions (**0.8289**). Weighted CE is favoured over the two tested alternative losses on this primary metric.

**Random Forest** remains the fixed classical reference rather than an architecture substitute. Its stronger observed Shuffling F1 provides a complementary baseline. TCN remains a Development comparator; it is not introduced as an additional qualified Test candidate.

**Seed 42** is the established training convention. Although its observed score is higher than the other tested seeds here, that observation is not the reason to select or alter the seed. The primary/reference pairing and fixed settings correspond to the existing historical protocol, not a new selection based on previously exposed Test results.

In [7]:
# ============= 06. Document the Primary and Reference Selection ==============

# --------- 1. Summarise the Observed Evidence and Recorded Decision ----------
overall_best = selection_comparison.iloc[0]
robustness_best = robustness_comparison.iloc[0]
rf_best_shuffling = shuffling.sort_values(
    "shuffling_pooled_f1", ascending=False
).iloc[0]
if (overall_best["model"] != FIXED_PRIMARY or
    (robustness_best["architecture"], robustness_best["loss_method"])
        != (FIXED_PRIMARY, "weighted_ce")):
    raise ValueError(
        "Current Development evidence differs from the documented primary "
        "selection. Review the scientific record before changing the model."
    )
if rf_best_shuffling["model"] != FIXED_REFERENCE:
    print("Highest Shuffling F1 belongs to:", rf_best_shuffling["model"])

selection_decisions = pd.DataFrame([
    {
        "role": "Primary model",
        "configuration": "1D-ResNet + inverse-sqrt Weighted CE",
        "source": "Notebook 10 and 11 Development evidence",
        "rationale": "Highest observed mean Macro F1",
    },
    {
        "role": "Fixed classical reference",
        "configuration": "Random Forest, 120 trees",
        "source": "Notebook 10 Development evidence",
        "rationale": "Fixed classical comparator; stronger Shuffling F1",
    },
    {
        "role": "Development neural comparator",
        "configuration": "TCN + inverse-sqrt Weighted CE",
        "source": "Notebook 11 robustness evidence",
        "rationale": "Seed sensitivity and reference architecture contrast",
    },
    {
        "role": "Exploratory ResNet loss alternatives",
        "configuration": "Focal and Class-Balanced Cross-Entropy",
        "source": "Notebook 11 paired loss evidence",
        "rationale": "Lower mean Macro F1; minority-class precision/recall trade-off",
    },
])
display(selection_decisions)
selection_decisions.to_csv(
    output_dir / "12_development_model_selection_decisions.csv",
    index=False,
)


,role,configuration,source,rationale
0,Primary model,1D-ResNet + inverse-sqrt Weighted CE,Notebook 10 and 11 Development evidence,Highest observed mean Macro F1
1,Fixed classical reference,"Random Forest, 120 trees",Notebook 10 Development evidence,Fixed classical comparator; stronger Shuffling F1
2,Development neural comparator,TCN + inverse-sqrt Weighted CE,Notebook 11 robustness evidence,Seed sensitivity and reference architecture co...
3,Exploratory ResNet loss alternatives,Focal and Class-Balanced Cross-Entropy,Notebook 11 paired loss evidence,Lower mean Macro F1; minority-class precision/...


**07. Record the Existing Source and Training Specification**

The source, preprocessing, primary-model and classical-reference settings below retain the established HARTH v2.0 protocol. They are recorded directly in this notebook, without requiring separate approval files or historical code comparisons. Notebook 10 and 11 provide Development selection evidence. Previously inspected held-out results are not used to change the selected models or settings.

The 1D-ResNet has six input channels, 32 hidden channels and eight classes; it uses Development-only channel standardisation and inverse-square-root weighted Cross-Entropy, Adam (learning rate 0.001), batch size 128, 400 updates and a final-step checkpoint. The Random Forest uses 30 deterministic features, 120 trees, minimum leaf size three, balanced-subsample class weights and seed 42.

The source revision, nominal sampling rate, label-pure timestamp-continuous windowing, participant-disjoint cohort and reserved S023/S032 identities remain explicit. Participant and window counts come from the current Notebook 10/11 evidence. The recorded settings are not new Test authorisation or independent reproduction of historical fitting.

In [ ]:
# ========= 07. Record the Existing Source and Training Specification =========

# -------------------- 1. Define the Established Source Recipe ----------------
source = {
    "dataset": "HARTH v2.0",
    "revision": "dad2cfbe89a26f72f19770419469ac037de200df",
    "sampling_rate_hz": 50,
    "window_size": 100,
    "step_size": 50,
    "annotation_purity": 1,
    "timestamp_continuity": True,
    "task_codes": list(SOURCE_CLASSES),
    "development_participants": development_people,
    "development_cap_per_person": 500,
    "expected_development_windows": development_windows,
    "reserved_test_people": ["S023", "S032"],
    "test_cap_per_person": 500,
    "unknown_source_codes": "Keep observed codes; exclude non-task windows",
}

# ------------------- 2. Define the Established Model Recipes -----------------
primary = {
    "architecture": FIXED_PRIMARY,
    "in_channels": 6,
    "hidden_channels": 32,
    "classes": list(SOURCE_CLASSES),
    "training_loss": "inverse_sqrt_weighted_ce",
    "random_seed": 42,
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "batch_size": 128,
    "optimizer_steps": 400,
    "checkpoint": "final_step",
    "preprocessing": "Development-only per-channel mean and SD",
}
reference_model = {
    "architecture": FIXED_REFERENCE,
    "n_estimators": 120,
    "min_samples_leaf": 3,
    "class_weight": "balanced_subsample",
    "seed": 42,
    "features": ["mean", "std", "rms", "peak_to_peak", "waveform_length"],
}
evaluation = {
    "main_metric": "Fixed-eight-class Macro F1",
    "additional": [
        "accuracy", "balanced_accuracy",
        "per-class precision/recall/F1/support",
        "per-person fixed-task Macro F1", "confusion matrices",
    ],
    "use_test_for_tuning": False,
    "temperature_scaling": False,
    "description": (
        "Qualified previously-inspected reserved Test, "
        "not fully blind external validation"
    ),
}

# ------------- 3. Export Source and Training Settings Once -------------------
source_specification = pd.DataFrame([
    ("Source", source["dataset"]),
    ("Pinned revision", source["revision"]),
    ("Sampling rate (Hz)", source["sampling_rate_hz"]),
    ("Window size (samples)", source["window_size"]),
    ("Stride (samples)", source["step_size"]),
    ("Label purity", source["annotation_purity"]),
    ("Timestamp continuity", source["timestamp_continuity"]),
    ("Source task codes", ", ".join(map(str, source["task_codes"]))),
    ("Development participants", source["development_participants"]),
    ("Development cap per person", source["development_cap_per_person"]),
    ("Development windows", source["expected_development_windows"]),
    ("Reserved participants", ", ".join(source["reserved_test_people"])),
    ("Reserved cap per person", source["test_cap_per_person"]),
    ("Other source codes", source["unknown_source_codes"]),
], columns=["property", "recorded_value"])
training_specification = pd.DataFrame([
    ("Primary architecture", primary["architecture"]),
    ("Input / hidden channels", f'{primary["in_channels"]} / {primary["hidden_channels"]}'),
    ("Class codes", ", ".join(map(str, primary["classes"]))),
    ("Loss", primary["training_loss"]),
    ("Optimiser", primary["optimizer"]),
    ("Learning rate", primary["learning_rate"]),
    ("Batch size", primary["batch_size"]),
    ("Optimiser updates", primary["optimizer_steps"]),
    ("Seed", primary["random_seed"]),
    ("Checkpoint", primary["checkpoint"]),
    ("Channel scaling", primary["preprocessing"]),
    ("Reference", reference_model["architecture"]),
    ("RF trees", reference_model["n_estimators"]),
    ("RF minimum leaf", reference_model["min_samples_leaf"]),
    ("RF class weighting", reference_model["class_weight"]),
    ("RF seed", reference_model["seed"]),
    ("RF features per channel", ", ".join(reference_model["features"])),
    ("Temperature scaling", evaluation["temperature_scaling"]),
], columns=["setting", "recorded_value"])
display(source_specification)
display(training_specification)
source_specification.to_csv(
    output_dir / "12_recorded_source_protocol.csv", index=False,
)
training_specification.to_csv(
    output_dir / "12_recorded_training_configuration.csv", index=False,
)


**08. Describe the Qualified Held-Out Reporting Scope**

The subsequent evaluation procedure specifies **fixed-eight-class Macro F1** as its primary metric, accompanied by accuracy, balanced accuracy, per-class precision/recall/F1/support, per-person fixed-task Macro F1 and confusion matrices. Both primary and reference models use the same source windows, class order and Development-fitted transforms.

The original reserved participants S023 and S032 were inspected descriptively in an earlier phase. Their later assessment is accordingly a **qualified, same-study held-out evaluation**, not a strictly blind external validation. The selected model and fixed source/training parameters are not modified using those participants.

Notebook 12 documents the established evaluation boundaries directly. It neither accesses Test recordings nor repeats the subsequent qualified evaluation.

**Calibration context:** Notebook 08 reported improved mean ECE and Brier score but worsened mean NLL after temperature scaling in a separate Development calibration experiment. Temperature scaling is not included in the recorded final recipe; these calibration results are not a matched assessment of the final fitted ResNet.

In [ ]:
# ============ 08. Describe the Qualified Held-Out Reporting Scope ============

# ------------------- 1. Read the Fixed Evaluation Contract -------------------
if evaluation["main_metric"] != "Fixed-eight-class Macro F1":
    raise ValueError("The recorded primary outcome metric has changed.")
if evaluation["use_test_for_tuning"]:
    raise ValueError("The recorded study does not permit Test-informed tuning.")
scope = pd.DataFrame([
    ("Primary metric", evaluation["main_metric"]),
    ("Secondary metrics", "; ".join(evaluation["additional"])),
    ("Compared configurations", "Primary 1D-ResNet and fixed Random Forest"),
    ("Preprocessing", "Development fitted only"),
    ("Checkpoint", primary["checkpoint"]),
    ("Reserved people", ", ".join(source["reserved_test_people"])),
    ("Historical exposure", "Reserved participants previously inspected"),
    ("Evaluation interpretation", evaluation["description"]),
    ("Model/threshold tuning using Test", "Not part of the protocol"),
], columns=["item", "recorded_rule"])
display(scope)
scope.to_csv(
    output_dir / "12_qualified_evaluation_scope.csv", index=False,
)


**09. Reconcile and Export the Development Selection Evidence**

This selection stage combines **7 benchmark model summaries**, **21 benchmark model–fold rows**, **4 robustness conditions**, **36 robustness model–fold rows** and the matching 29-person, 14,201-window Development cohort. It preserves the original model selection and evaluation procedure while providing a concise set of selection-specific CSV tables and one complementary scientific figure.

Final consistency checks compare saved fold means with their summaries, verify complete participant-fold coverage, confirm recorded class support, and ensure all selection metrics are finite and within valid ranges. These checks establish agreement between saved Development evidence tables, not independent reproduction from the original recordings.

No model is trained, no source recordings are loaded and no administrative approval record is issued by this notebook. The `12_` files consolidate Development-only selection evidence; final fitted models and qualified Test scores belong to the separate study stage.

In [ ]:
# ======== 09. Reconcile and Export the Development Selection Evidence ========

# ------------- 1. Reconcile Fold-Level and Class-Level Evidence --------------
expected_benchmark_folds = len(expected_models) * 3
expected_robust_folds = len(condition_set) * len(seed_fold_grid)
if len(folds_10) != expected_benchmark_folds:
    raise ValueError("Seven-model Development fold table is incomplete.")
if len(folds_11) != expected_robust_folds:
    raise ValueError("Multi-seed Development fold table is incomplete.")
if not all(int(g["validation_windows"].sum()) == development_windows
           for _, g in folds_10.groupby("model")):
    raise ValueError("Benchmark participant-fold windows do not cover the cohort.")
if not all(int(g["validation_windows"].sum()) == development_windows
           for _, g in folds_11.groupby(
               ["architecture", "loss_method", "seed"]
           )):
    raise ValueError("Robustness participant-fold windows do not cover the cohort.")
if not all(int(g["support"].sum()) == development_windows
           for _, g in classes.groupby("model")):
    raise ValueError("Benchmark class support does not cover all windows.")
if not all(int(g["support"].sum()) == development_windows
           for _, g in robust_classes.groupby(
               ["architecture", "loss_method", "seed"]
           )):
    raise ValueError("Robustness class support does not cover all windows.")
if len(tables["benchmark_fold_design"]) != 3 or int(
    tables["benchmark_fold_design"]["validation_windows"].sum()
) != development_windows:
    raise ValueError("Current Notebook 10 fold design differs in window support.")
for _, row in benchmark.iterrows():
    fold_average = folds_10.loc[
        folds_10["model"].eq(row["model"]), "macro_f1"
    ].mean()
    if not np.isclose(fold_average, row["macro_f1_mean"], atol=1e-10):
        raise ValueError(f"Benchmark fold mean differs for {row['model']}.")
for table, key in (
    (benchmark, "macro_f1_mean"),
    (robust, "mean_across_seed_fold_macro_f1"),
    (classes, "f1"),
    (robust_classes, "f1"),
):
    values = pd.to_numeric(table[key], errors="raise").to_numpy()
    if not np.isfinite(values).all() or not np.all(
        (0 <= values) & (values <= 1)
    ):
        raise ValueError(f"Invalid reported metric in {key}.")

# ---------- 2. Summarise the Selection Evidence and Package Outputs ----------
print("Development participants:", development_people)
print("Matched source windows:", development_windows)
print("Compared benchmark models:", len(benchmark))
print("Benchmark model–fold records:", len(folds_10))
print("Robustness model–fold records:", len(folds_11))
print("Selected primary:", primary["architecture"], "|", primary["training_loss"])
print("Classical reference:", reference_model["architecture"])
print("Source and training settings: documented in current 12_ CSV tables")
print("Output directory:", output_dir)
download_outputs_from_colab(
    folder="wearable_activity_recognition/output", stage="12"
)
